# 03 – Privacy, GDPR & Governance Controls


This notebook evaluates the dataset from a privacy and regulatory compliance perspective.

We identify all personally identifiable information (PII), assess governance gaps, and demonstrate privacy-enhancing techniques in alignment with GDPR principles and AI governance requirements.


In [ ]:
# We import Libraries 
import pandas as pd 
import numpy as np
import json 

In [ ]:
# We now load the cleaned dataset produced in 01 - data - quality.ipynb
# This ensures the privacy analysis uses the same cleaned data as the data-quality notebook

df_clean = pd.read_csv("../data/cleaned_dataset.csv")

print(f"Total cleaned records: {len(df_clean)}")
df_clean.head()

Total cleaned records: 500


,spending_behavior,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,decision.interest_rate,decision.approved_amount,financials.income_unified
0,"[{'category': 'Fitness', 'amount': 576}]",Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,female,1986-05-27,90230.0,37.0,0.42,0.0,False,high_dti_ratio,NaN,NaN,102000.0
1,"[{'category': 'Education', 'amount': 533}]",Kevin Roberts,kevin.roberts9@protonmail.com,992-61-4010,172.19.95.144,male,1999-08-01,10020.0,5.0,0.36,18200.0,False,algorithm_risk_score,NaN,NaN,41000.0
2,"[{'category': 'Healthcare', 'amount': 450}]",Lisa Gonzalez,lisa.gonzalez51@yahoo.com,833-33-5929,172.21.35.195,female,1982-08-24,90213.0,74.0,0.43,7090.0,True,NaN,3.4,76000.0,65000.0
3,"[{'category': 'Transportation', 'amount': 329}...",Karen Nelson,karen.nelson35@outlook.com,486-50-5539,172.31.79.76,female,NaN,90217.0,9.0,0.41,10327.0,False,high_dti_ratio,NaN,NaN,69000.0
4,"[{'category': 'Insurance', 'amount': 585}]",Christine Mitchell,christine.mitchell3@outlook.com,400-91-8156,172.25.44.173,female,NaN,90296.0,76.0,0.06,15011.0,False,algorithm_risk_score,NaN,NaN,39000.0


## Dataset Overview

The cleaned dataset produced in the data-quality notebook is used as the starting point for the privacy and governance analysis.

This dataset still contains several attributes that may directly or indirectly identify individuals. Under GDPR, these fields are considered personally identifiable information (PII) and require careful handling to ensure data protection and regulatory compliance.

In [ ]:
# We identify personally identifiable information (PII) columns

pii_columns = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth",
    "applicant_info.zip_code",
]

df_clean[pii_columns].head()

,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.date_of_birth,applicant_info.zip_code
0,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,1986-05-27,90230.0
1,Kevin Roberts,kevin.roberts9@protonmail.com,992-61-4010,172.19.95.144,1999-08-01,10020.0
2,Lisa Gonzalez,lisa.gonzalez51@yahoo.com,833-33-5929,172.21.35.195,1982-08-24,90213.0
3,Karen Nelson,karen.nelson35@outlook.com,486-50-5539,172.31.79.76,NaN,90217.0
4,Christine Mitchell,christine.mitchell3@outlook.com,400-91-8156,172.25.44.173,NaN,90296.0


## Identifying Personally Identifiable Information (PII)

The dataset contains several attributes that can directly identify individuals, such as full name, email, Social Security Number (SSN), and IP address.

It also includes quasi-identifiers such as date of birth and zip code, which can potentially be used to re-identify individuals when combined with other information.

Identifying these fields is essential to evaluate privacy risks and to apply privacy-preserving techniques such as pseudonymization and data minimization.

In [ ]:
# EMAIL HASH

import hashlib

def hash_email(email):
    if pd.isna(email):
        return email
    return hashlib.sha256(email.encode()).hexdigest()

df_clean["email_hash"] = df_clean["applicant_info.email"].apply(hash_email)

## Email Pseudonymization

To reduce privacy risks, email addresses are transformed using SHA-256 hashing.

Hashing converts the original email value into a secure representation that cannot easily be reversed. This allows the dataset to retain analytical usefulness while protecting the identity of individuals.

This technique is an example of pseudonymization, which is recommended under GDPR to reduce the risks associated with processing personal data.

In [ ]:
# SSN

df_clean["ssn_masked"] = df_clean["applicant_info.ssn"].str.replace(
    r"\d{3}-\d{2}",
    "***-**",
    regex=True
)

## Protection of Sensitive Identifiers

The Social Security Number (SSN) is a highly sensitive personal identifier.

To reduce the risk of identity exposure, the first digits of the SSN are masked. This transformation limits the ability to identify individuals while still allowing partial verification if required.

Masking sensitive identifiers helps reduce the potential harm associated with data breaches or unauthorized access.

In [ ]:
# YEAR 

df_clean["dob_year"] = pd.to_datetime(
    df_clean["applicant_info.date_of_birth"], errors="coerce"
).dt.year

## Data Minimization

Under GDPR principles, organizations should only collect and store personal data that is strictly necessary for the intended purpose.

Instead of storing the full date of birth, the dataset retains only the year of birth. This allows certain analytical insights, such as age estimation, while significantly reducing the level of personal information stored.

This transformation illustrates the principle of data minimization.

In [ ]:
# BEFORE vs. AFTER

df_clean[[
    "applicant_info.email", "email_hash",
    "applicant_info.ssn", "ssn_masked",
    "applicant_info.date_of_birth", "dob_year"
]].head()

,applicant_info.email,email_hash,applicant_info.ssn,ssn_masked,applicant_info.date_of_birth,dob_year
0,stephanie.nguyen47@mail.com,137c1b4a89343995539bffbc9ce6edaa65bcb498fc0434...,427-90-1892,***-**-1892,1986-05-27,1986.0
1,kevin.roberts9@protonmail.com,ad586ff31ede0ddcf626645f064f888e0f8e7beb3f81e2...,992-61-4010,***-**-4010,1999-08-01,1999.0
2,lisa.gonzalez51@yahoo.com,f110fede2c1f19f308c6cbf664f079f53438e00dfe44d6...,833-33-5929,***-**-5929,1982-08-24,1982.0
3,karen.nelson35@outlook.com,a094d9056654b513ce0e2f5c254f33470604a4549c72d3...,486-50-5539,***-**-5539,NaN,NaN
4,christine.mitchell3@outlook.com,1a407bcae335e67007abb52c9b1766b7b9cd78403829ff...,400-91-8156,***-**-8156,NaN,NaN


## Privacy Transformations

The table above illustrates how sensitive attributes are transformed.

Direct identifiers such as emails and SSNs are pseudonymized or masked, while detailed personal attributes like full birth dates are reduced to less sensitive representations.

These transformations help maintain analytical value while significantly reducing the risk of identifying individuals.

In [ ]:
# We create a privacy-friendly version of the dataset by dropping direct identifiers

df_privacy = df_clean.drop(columns=[
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth",
])
import os
os.makedirs('../data', exist_ok=True) 

df_privacy.to_csv("../data/cleaned_privacy_dataset.csv", index=False)
df_privacy.to_csv("../data/cleaned_privacy_dataset.csv", index=False)

## Creating a Privacy-Friendly Dataset

A privacy-friendly version of the dataset is created by removing direct identifiers such as names, emails, SSNs, IP addresses, and full dates of birth.

This approach follows the principle of Privacy by Design, where sensitive personal information is removed whenever it is not required for analytical purposes.

The resulting dataset retains useful features for analysis while significantly reducing the risk of re-identification.

## Governance Risk: Missing Processing Timestamps

During the data quality analysis, we identified a significant governance issue related to missing processing timestamps.

Timestamp missingness undermines model traceability and version control. Without reliable timestamps, it becomes difficult to determine when a credit decision was made, which model version was used, or whether policy changes occurred over time.

To mitigate this risk, organizations should implement automated logging at the model inference stage and enforce database-level constraints such as NOT NULL for timestamp fields. These measures help ensure proper audit trails and strengthen regulatory accountability.

## Conclusion

The privacy analysis demonstrates how sensitive personal data can be protected through pseudonymization and data minimization techniques.

By identifying PII and applying privacy-preserving transformations, the dataset becomes significantly safer to use while maintaining its analytical value.

These measures support compliance with GDPR principles such as data minimization, integrity and confidentiality of processing, and responsible handling of personal information.